# Text features for ECOSOC transcripts

Reads `data/interim/transcripts.parquet`, produces three feature tiers per `(year, segment)` block:

- **Tier 1**: length, sentiment (VADER), lexicon hits across 7 thematic categories.
- **Tier 2**: TF-IDF (`max_features=2000`, 1–2 grams).
- **Tier 3**: sentence-transformer embeddings (`all-MiniLM-L6-v2`), mean-pooled.

Procedure-boilerplate sentences are filtered before featurization.

## 0. Setup

In [1]:
!pip install pandas pyarrow scikit-learn vaderSentiment sentence-transformers

In [2]:
import re
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
TRANSCRIPTS = ROOT / "data" / "interim" / "transcripts.parquet"
FEAT_DIR = ROOT / "data" / "features"
FEAT_DIR.mkdir(parents=True, exist_ok=True)

AGG_KEYS = ["year", "segment"]   # change to ["year"] to collapse segments
RNG = 42
print("transcripts:", TRANSCRIPTS)

transcripts: /Users/latahviawilliams/Downloads/Big_Data_export/final_project/ecosoc/data/interim/transcripts.parquet


## 1. Load + boilerplate filter

Summary records have heavy procedural boilerplate ("The President", "adopted without a vote", "by N votes to M"). Drop those sentences before featurizing.

In [3]:
BOILERPLATE_PATTERNS = [
    r"^\s*The President\b",
    r"^\s*The Acting President\b",
    r"^\s*The Chair(person|man|woman)?\b",
    r"^\s*The (Vice-)?President\b",
    r"^\s*The representative of .* spoke\b",
    r"^\s*The meeting (was )?(called to order|adjourned|suspended|resumed)\b",
    r"^\s*adopted without a vote\b",
    r"^\s*by \d+ votes to \d+",
    r"^\s*Agenda item\b",
    r"^\s*Draft (resolution|decision)\b",
    r"^\s*United Nations\s*$",
    r"^\s*E/\d{4}/SR\.\d+\b",
]
BOILERPLATE_RE = re.compile("|".join(BOILERPLATE_PATTERNS), re.IGNORECASE)

SENT_SPLIT_RE = re.compile(r"(?<=[.!?])\s+(?=[A-Z(])")

def clean_text(t: str) -> str:
    if not isinstance(t, str) or not t.strip():
        return ""
    sents = SENT_SPLIT_RE.split(t)
    keep = [s for s in sents if s and not BOILERPLATE_RE.search(s)]
    return " ".join(keep)

raw = pd.read_parquet(TRANSCRIPTS)
raw["text_clean"] = raw.text.fillna("").map(clean_text)
raw["n_chars_clean"] = raw.text_clean.str.len()
print("rows:", len(raw),
      "| empty after clean:", int((raw.n_chars_clean < 200).sum()),
      "| mean chars before/after:", int(raw.n_chars.mean()), int(raw.n_chars_clean.mean()))
raw[["meeting_id", "n_chars", "n_chars_clean"]].head()

rows: 910 | empty after clean: 4 | mean chars before/after: 37364 35601


,meeting_id,n_chars,n_chars_clean
0,E/2000/SR.3,15766,12076
1,E/2000/SR.4,32904,32226
2,E/2001/SR.1,21294,20901
3,E/2001/SR.2,16143,10461
4,E/2001/SR.3,20828,20604


## 2. Aggregate to (year, segment) blocks

In [4]:
blocks = (raw[raw.n_chars_clean >= 200]
          .groupby(AGG_KEYS, as_index=False)
          .agg(text_clean=("text_clean", " ".join),
               n_meetings=("meeting_id", "nunique"),
               n_chars=("n_chars_clean", "sum")))
print("blocks:", len(blocks))
blocks.head()

blocks: 26


,year,segment,text_clean,n_meetings,n_chars
0,2000,SR,United Nations E/2000/SR.3\n \nEconomic and So...,2,44302
1,2001,SR,United Nations E/2001/SR.1\n \nEconomic and So...,32,967226
2,2002,SR,United Nations E/2002/SR.1\n \nEconomic and So...,44,1587328
3,2003,SR,United Nations E/2003/SR.1\n \nEconomic and So...,45,1242404
4,2004,SR,United Nations E/2004/SR.1\n \nEconomic and So...,54,1711165


## 3. Tier 1 — length / sentiment / lexicon

In [5]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

LEXICONS = {
    "development": {"development", "sustainable", "poverty", "growth", "infrastructure",
                    "capacity", "sdg", "sdgs"},
    "humanitarian": {"humanitarian", "emergency", "refugees", "displacement", "famine",
                     "hunger", "crisis", "relief"},
    "climate": {"climate", "emissions", "adaptation", "mitigation", "biodiversity",
                "renewable", "environment", "environmental"},
    "gender": {"gender", "women", "girls", "equality", "reproductive", "maternal"},
    "conflict": {"conflict", "war", "violence", "peace", "security", "ceasefire"},
    "health": {"health", "pandemic", "vaccine", "vaccination", "hiv", "aids", "disease",
               "hospital", "who"},
    "governance": {"governance", "institutions", "corruption", "rule", "law", "democracy",
                   "transparency", "accountability"},
}

TOK_RE = re.compile(r"[A-Za-z][A-Za-z\-]+")

def lex_counts(text: str) -> dict:
    toks = [t.lower() for t in TOK_RE.findall(text)]
    n = max(len(toks), 1)
    out = {f"lex_{k}": sum(t in lex for t in toks) / n for k, lex in LEXICONS.items()}
    out["n_tokens"] = len(toks)
    return out

vader = SentimentIntensityAnalyzer()

def vader_compound(text: str) -> float:
    # truncate to keep VADER fast
    return vader.polarity_scores(text[:50_000])["compound"]

tier1_rows = []
for _, r in blocks.iterrows():
    feats = lex_counts(r.text_clean)
    feats["sentiment"] = vader_compound(r.text_clean)
    feats.update({k: r[k] for k in AGG_KEYS})
    feats["n_meetings"] = r.n_meetings
    tier1_rows.append(feats)
tier1 = pd.DataFrame(tier1_rows)
tier1 = tier1[AGG_KEYS + ["n_meetings", "n_tokens", "sentiment"] +
              [c for c in tier1.columns if c.startswith("lex_")]]
t1_path = FEAT_DIR / "tier1.parquet"
tier1.to_parquet(t1_path, index=False)
print(f"wrote {t1_path} ({len(tier1)} rows)")
tier1.head()

wrote /Users/latahviawilliams/Downloads/Big_Data_export/final_project/ecosoc/data/features/tier1.parquet (26 rows)


,year,segment,n_meetings,n_tokens,sentiment,lex_development,lex_humanitarian,lex_climate,lex_gender,lex_conflict,lex_health,lex_governance
0,2000,SR,2,6671,0.9999,0.006746,0.004497,0.000000,0.000300,0.008694,0.029681,0.000450
1,2001,SR,32,139827,1.0000,0.012530,0.003719,0.000930,0.001452,0.003140,0.004270,0.002317
2,2002,SR,44,228226,1.0000,0.013714,0.004684,0.001122,0.002493,0.002866,0.005205,0.002134
3,2003,SR,45,179289,1.0000,0.013771,0.004395,0.000837,0.001601,0.002231,0.002694,0.002265
4,2004,SR,54,246626,1.0000,0.013673,0.003949,0.001070,0.003929,0.002607,0.001898,0.002777


## 4. Tier 2 — TF-IDF

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse as sp
import joblib

tfidf = TfidfVectorizer(max_features=2000, ngram_range=(1, 2),
                        min_df=3, max_df=0.95,
                        sublinear_tf=True, stop_words="english")
X = tfidf.fit_transform(blocks.text_clean)
print("TF-IDF shape:", X.shape)

tier2 = pd.DataFrame(X.toarray(), columns=[f"t_{w}" for w in tfidf.get_feature_names_out()])
for k in AGG_KEYS:
    tier2.insert(0, k, blocks[k].values)
t2_path = FEAT_DIR / "tier2_tfidf.parquet"
tier2.to_parquet(t2_path, index=False)
joblib.dump(tfidf, FEAT_DIR / "tfidf_vectorizer.joblib")
sp.save_npz(FEAT_DIR / "tier2_tfidf.npz", X)
print(f"wrote {t2_path} (+ vectorizer + npz)")

TF-IDF shape: (26, 2000)
wrote /Users/latahviawilliams/Downloads/Big_Data_export/final_project/ecosoc/data/features/tier2_tfidf.parquet (+ vectorizer + npz)


## 5. Tier 3 — sentence-transformer embeddings

We chunk each block to ~256-token windows, encode with `all-MiniLM-L6-v2` (384-dim), and mean-pool back to one vector per `(year, segment)`. Cached to disk so reruns are fast.

In [7]:
RUN_TIER3 = True   # set False to skip the heavy step
EMB_MODEL = "all-MiniLM-L6-v2"
CHUNK_WORDS = 200

if RUN_TIER3:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(EMB_MODEL)

    def chunks(text: str, n: int = CHUNK_WORDS):
        words = text.split()
        for i in range(0, len(words), n):
            yield " ".join(words[i:i + n])

    block_vecs = []
    for _, r in blocks.iterrows():
        cs = list(chunks(r.text_clean)) or [r.text_clean[:1000]]
        emb = model.encode(cs, batch_size=32, show_progress_bar=False,
                           convert_to_numpy=True, normalize_embeddings=True)
        block_vecs.append(emb.mean(axis=0))
    emb_mat = np.vstack(block_vecs)
    tier3 = pd.DataFrame(emb_mat, columns=[f"e_{i}" for i in range(emb_mat.shape[1])])
    for k in AGG_KEYS:
        tier3.insert(0, k, blocks[k].values)
    t3_path = FEAT_DIR / "tier3_embeddings.parquet"
    tier3.to_parquet(t3_path, index=False)
    np.save(FEAT_DIR / "tier3_embeddings.npy", emb_mat)
    print(f"wrote {t3_path}, shape={emb_mat.shape}")
else:
    print("Tier 3 skipped (RUN_TIER3=False)")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

wrote /Users/latahviawilliams/Downloads/Big_Data_export/final_project/ecosoc/data/features/tier3_embeddings.parquet, shape=(26, 384)


## 6. Quick QA — top tokens per year

In [8]:
vocab = np.array(tfidf.get_feature_names_out())
X_arr = X.toarray()
for i, row in blocks.iterrows():
    top_idx = X_arr[i].argsort()[::-1][:8]
    label = " / ".join(str(row[k]) for k in AGG_KEYS)
    print(f"{label}: ", ", ".join(vocab[top_idx]))

2000 / SR:  unaids, 2000, pandemic, epidemic, portugal, spread, peacekeeping, draft decision
2001 / SR:  page, cca, 2001, 4108, 4108 palais, room 4108, nations geneva, palais des
2002 / SR:  cca, commission human, 2002, record submitted, bhutan, monterrey, monterrey consensus, millennium declaration
2003 / SR:  ldcs, 2003, page, room 4108, 4108 palais, 4108, bissau, guinea bissau
2004 / SR:  cca, ldcs, brussels programme, 2004, commission human, bhutan, bissau, guinea bissau
2005 / SR:  2005, bissau, guinea bissau, tsunami, commission human, millennium, millennium development, millennium declaration
2006 / SR:  ldcs, page, 4108, 4108 palais, room 4108, bissau, guinea bissau, palais des
2007 / SR:  2007, ministerial review, 4108, 4108 palais, room 4108, annual ministerial, page, des nations
2008 / SR:  08, rking languages, submitted wo, mdgs, record shoul, shoul submitted, 2008, global food
2009 / SR:  2009, mdgs, rking languages, submitted wo, ldcs, shoul submitted, record shoul, minis